In [45]:
# Install library (jalankan sekali saja)
!pip install Sastrawi
import pandas as pd
import numpy as np
import re
import string

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [46]:
!pip install Sastrawi nltk scikit-learn wordcloud

In [47]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
df = pd.read_csv('/content/drive/MyDrive/dokumen magang/dataset_hasil_eda (2).csv')

In [49]:
df.head()

,id,pengaduan,kategori,jumlah_karakter,jumlah_kata,prioritas
0,1,"Mohon, jalan berlubang di RT 01, udah seminggu.",Infrastruktur,47,8,Sedang
1,2,"Pak, trotoar rusak di Jl. Mawar, udah 3 hari.",Infrastruktur,45,9,Sedang
2,3,"Bu, drainase mampet depan rumah, udah 3 hari.",Infrastruktur,45,8,Sedang
3,4,"Saya mau lapor, selokan kotor di RT 02, ganggu...",Infrastruktur,57,10,Sedang
4,5,"Pak, lampu jalan mati di Gang Melati, mohon se...",Infrastruktur,60,10,Sedang


In [50]:
df.columns

Index(['id', 'pengaduan', 'kategori', 'jumlah_karakter', 'jumlah_kata',
       'prioritas'],
      dtype='object')

In [51]:
##kolom yang akan diproses
df = df[['pengaduan']].copy()

In [52]:
df.head()

,pengaduan
0,"Mohon, jalan berlubang di RT 01, udah seminggu."
1,"Pak, trotoar rusak di Jl. Mawar, udah 3 hari."
2,"Bu, drainase mampet depan rumah, udah 3 hari."
3,"Saya mau lapor, selokan kotor di RT 02, ganggu..."
4,"Pak, lampu jalan mati di Gang Melati, mohon se..."


In [53]:
##Case Folding

##Mengubah semua huruf menjadi huruf kecil.
def case_folding(text):
    text = text.lower()
    return text

df['case_folding'] = df['pengaduan'].apply(case_folding)

In [54]:
##Menghapus URL, angka, tanda baca, emoji, dan karakter khusus.
def cleaning(text):

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"www\S+", "", text)

    text = re.sub(r"\d+", "", text)

    text = re.sub(r"[^\w\s]", " ", text)

    text = re.sub(r"_", " ", text)

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

df['cleaning'] = df['case_folding'].apply(cleaning)

In [55]:
normalisasi = {

    "gk":"tidak",
    "ga":"tidak",
    "nggak":"tidak",
    "tdk":"tidak",

    "yg":"yang",
    "dr":"dari",
    "dgn":"dengan",

    "udh":"sudah",
    "blm":"belum",

    "krn":"karena",

    "bgt":"banget",

    "tp":"tetapi",

    "aja":"saja",

    "org":"orang",

    "sy":"saya",

    "utk":"untuk"

}

In [56]:
##Fungsi normalisasi
def normalisasi_kata(text):

    kata = text.split()

    hasil = [normalisasi[word] if word in normalisasi else word
             for word in kata]

    return " ".join(hasil)

df['normalisasi'] = df['cleaning'].apply(normalisasi_kata)

In [57]:
##Tokenizing
def tokenizing(text):
    return text.split()

df['token'] = df['normalisasi'].apply(tokenizing)

In [58]:
##Stopword Bahasa Indonesia.

stop_words = stopwords.words('indonesian')

In [59]:
stop_words.extend([
    "nya",
    "nih",
    "dong",
    "tolong",
    "pak",
    "bu"
])

##Fungsi

def stopword_removal(tokens):

    hasil = [word for word in tokens if word not in stop_words]




    return hasil

df['stopword'] = df['token'].apply(stopword_removal)

In [60]:
##Stemming
factory = StemmerFactory()

stemmer = factory.create_stemmer()
def stemming(tokens):

    hasil = [stemmer.stem(word) for word in tokens]

    return hasil

df['stemming'] = df['stopword'].apply(stemming)

In [61]:
##Menggabungkan token menjadi kalimat kembali.

def join_text(tokens):

    return " ".join(tokens)

df['pengaduan_clean'] = df['stemming'].apply(join_text)

In [63]:
df = df[df['pengaduan_clean'] != ""]

##Reset index.

df.reset_index(drop=True, inplace=True)

In [66]:
df = df.drop_duplicates(subset='pengaduan_clean')

##Reset index lagi.

df.reset_index(drop=True, inplace=True)

In [68]:
df[['pengaduan','pengaduan_clean']].head(20)

,pengaduan,pengaduan_clean
0,"Mohon, jalan berlubang di RT 01, udah seminggu.",mohon jalan lubang rt udah minggu
1,"Pak, trotoar rusak di Jl. Mawar, udah 3 hari.",trotoar rusak jl mawar udah
2,"Bu, drainase mampet depan rumah, udah 3 hari.",drainase mampet rumah udah
3,"Saya mau lapor, selokan kotor di RT 02, ganggu...",lapor selokan kotor rt ganggu aktivitas
4,"Pak, lampu jalan mati di Gang Melati, mohon se...",lampu jalan mati gang melati mohon tindak
5,"Mohon, trotoar rusak depan rumah, udah seminggu.",mohon trotoar rusak rumah udah minggu
6,"Mohon, trotoar rusak di Jl. Mawar, bikin warga...",mohon trotoar rusak jl mawar bikin warga resah
7,"Mohon, drainase mampet di RT 01, tolong dicek.",mohon drainase mampet rt cek
8,"Bu, air bersih sering mati depan rumah, mohon ...",air bersih mati rumah mohon tindak
9,"Mohon, trotoar rusak di RT 01, bikin warga resah.",mohon trotoar rusak rt bikin warga resah


In [69]:
df.to_csv(
    "/content/drive/MyDrive/dokumen magang/pengaduan_preprocessing.csv",
    index=False
)

# 3.X Text Preprocessing

## 3.X.1 Pengertian Text Preprocessing

Text preprocessing merupakan tahapan awal dalam pengolahan data teks yang bertujuan untuk membersihkan dan menyeragamkan data sebelum digunakan pada proses klasifikasi. Data pengaduan masyarakat yang berasal dari pengguna umumnya masih mengandung berbagai variasi penulisan, seperti penggunaan huruf kapital, singkatan, tanda baca, angka, karakter khusus, serta kata-kata yang tidak memiliki makna penting dalam proses klasifikasi.

Pada penelitian ini, text preprocessing dilakukan untuk menghasilkan data teks yang lebih bersih dan konsisten sehingga dapat meningkatkan kualitas fitur yang akan diekstraksi menggunakan metode TF-IDF dan meningkatkan performa model klasifikasi.

---

## 3.X.2 Tahapan Text Preprocessing

Tahapan text preprocessing yang diterapkan pada penelitian ini terdiri dari beberapa proses sebagai berikut.

### 1. Case Folding

Case folding merupakan proses mengubah seluruh huruf pada teks menjadi huruf kecil (lowercase). Proses ini bertujuan agar tidak terdapat perbedaan antara huruf kapital dan huruf kecil sehingga setiap kata memiliki bentuk yang seragam.

Contoh:

Sebelum:

```
Lampu Jalan Mati di RT 03
```

Sesudah:

```
lampu jalan mati di rt 03
```

---

### 2. Cleaning

Tahap cleaning dilakukan untuk menghapus karakter-karakter yang tidak diperlukan dalam proses analisis teks. Karakter yang dihapus meliputi URL, angka, tanda baca, karakter khusus, simbol, serta spasi berlebih.

Contoh:

Sebelum:

```
Lampu jalan mati!!! Tolong dong 😭😭
```

Sesudah:

```
lampu jalan mati tolong dong
```

Tahapan ini bertujuan agar data hanya berisi kata-kata yang memiliki informasi penting.

---

### 3. Normalisasi Kata

Normalisasi merupakan proses mengubah kata-kata tidak baku atau singkatan menjadi bentuk bakunya. Pada data pengaduan masyarakat banyak ditemukan penggunaan bahasa sehari-hari seperti "gk", "yg", "udh", dan "blm". Kata-kata tersebut diubah menjadi bentuk baku agar memiliki representasi yang sama.

Contoh:

| Kata Tidak Baku | Kata Baku |
| --------------- | --------- |
| gk              | tidak     |
| yg              | yang      |
| udh             | sudah     |
| blm             | belum     |
| krn             | karena    |
| dr              | dari      |

Contoh hasil normalisasi:

Sebelum:

```
lampu jalan gk nyala dr kmrn
```

Sesudah:

```
lampu jalan tidak nyala dari kemarin
```

---

### 4. Tokenizing

Tokenizing merupakan proses memecah sebuah kalimat menjadi kumpulan kata (token). Setiap kata dipisahkan sehingga dapat diproses lebih lanjut pada tahap berikutnya.

Contoh:

Sebelum:

```
lampu jalan mati
```

Sesudah:

```
["lampu", "jalan", "mati"]
```

---

### 5. Stopword Removal

Stopword removal merupakan proses menghapus kata-kata umum yang memiliki kontribusi kecil terhadap proses klasifikasi, seperti "yang", "dan", "di", "ke", dan "dari". Penghapusan stopword bertujuan agar model lebih fokus pada kata-kata yang memiliki informasi penting.

Contoh:

Sebelum:

```
lampu jalan mati di depan rumah
```

Sesudah:

```
lampu jalan mati depan rumah
```

---

### 6. Stemming

Stemming merupakan proses mengubah kata berimbuhan menjadi bentuk kata dasar menggunakan pustaka Sastrawi. Proses ini bertujuan mengurangi variasi kata yang memiliki makna sama.

Contoh:

| Sebelum    | Sesudah |
| ---------- | ------- |
| diperbaiki | baik    |
| mengganggu | ganggu  |
| pelayanan  | layan   |
| laporan    | lapor   |

Contoh:

Sebelum:

```
jalan sedang diperbaiki
```

Sesudah:

```
jalan sedang baik
```

---

### 7. Join Token

Setelah proses tokenizing, stopword removal, dan stemming selesai dilakukan, seluruh token digabungkan kembali menjadi sebuah kalimat. Kolom hasil inilah yang digunakan pada proses ekstraksi fitur menggunakan metode TF-IDF.

Contoh:

Sebelum:

```
["lampu", "jalan", "mati"]
```

Sesudah:

```
lampu jalan mati
```

---

### 8. Penghapusan Data Kosong dan Data Duplikat

Tahap terakhir pada preprocessing adalah menghapus data yang memiliki hasil preprocessing kosong serta menghapus data duplikat. Proses ini bertujuan agar dataset yang digunakan pada tahap pelatihan model memiliki kualitas yang lebih baik dan tidak menimbulkan bias.

---

## 3.X.3 Hasil Text Preprocessing

Hasil text preprocessing menghasilkan dataset yang lebih bersih dan siap digunakan pada tahap pelabelan kategori, pelabelan prioritas, serta ekstraksi fitur menggunakan metode TF-IDF. Pada penelitian ini, hasil preprocessing disimpan dalam file **pengaduan_preprocessing.csv**.

Dataset hasil preprocessing terdiri atas beberapa kolom, yaitu:

* pengaduan
* case_folding
* cleaning
* normalisasi
* token
* stopword
* stemming
* pengaduan_clean

Kolom **pengaduan_clean** merupakan hasil akhir preprocessing dan digunakan sebagai masukan (input) pada proses ekstraksi fitur TF-IDF serta pelatihan model klasifikasi kategori dan prioritas.

Secara keseluruhan, proses text preprocessing berhasil mengubah data pengaduan yang semula masih mengandung berbagai variasi penulisan menjadi data yang lebih bersih, seragam, dan siap digunakan pada tahapan machine learning berikutnya.
